# Import Libraries:

In [61]:
import numpy as np
import pandas as pd
import re
from contractions import fix 
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
# nltk.download('stopwords')
from sklearn.feature_extraction.text import CountVectorizer 
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import joblib

In [23]:
# Increase the max column width
pd.set_option('display.max_colwidth', None)

# Load the data:

In [ ]:
df = pd.read_csv("data/spam.csv", encoding='latin1')
df.head(3)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN


# Removing unnecessary columns:

In [5]:
df=df.drop(columns=['Unnamed: 2',	'Unnamed: 3',	'Unnamed: 4'], axis=0)

In [6]:
df.head(3)

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...


# change the names of the columns to more descriptive

In [11]:
df.rename(columns={'v1':'target', 'v2':'message'}, inplace=True)

# Understanding the data:

In [16]:
print(f"Number of Rows: {df.shape[0]}, Number of Columns: {df.shape[1]}\n")
print(f"\nColumn info of the dataset:\n{df.info()}\n")
print(f"\nTarget count: \n{df['target'].value_counts()}\n")

Number of Rows: 5572, Number of Columns: 2

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   target   5572 non-null   object
 1   message  5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB

Column info of the dataset:
None


Target count: 
target
ham     4825
spam     747
Name: count, dtype: int64



# From the above we can say that:
1. There is no null values in the dataset..
2. Dataset is heavily imbalanced..

ham % = 4825/5572*100 = 86.5%

spam % = 747/5572*100 = 13.5%

# Convert Target to numerical feature..

In [19]:
df['target'] = df['target'].replace(['ham', 'spam'], [0,1])

C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_33468\3737512646.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['target'] = df['target'].replace(['ham', 'spam'], [0,1])


In [28]:
df.head(20)

,target,message
0,0,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives around here though"
5,1,"FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, å£1.50 to rcv"
6,0,Even my brother is not like to speak with me. They treat me like aids patent.
7,0,As per your request 'Melle Melle (Oru Minnaminunginte Nurungu Vettam)' has been set as your callertune for all Callers. Press *9 to copy your friends Callertune
8,1,WINNER!! As a valued network customer you have been selected to receivea å£900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.
9,1,Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030


# Text Preprocessing:

1. Replace email addresses with 'emailaddr'
2. Replace URLs with 'httpaddr'
3. Replace money symbols with 'moneysymb'
4. Replace phone numbers with 'phonenumbr'
5. Replace numbers with 'numbr'

In [36]:
#  Method 1: using domain knowledge to guide the model.. 
spam_words = ['winner', 'free', 'claim', 'valid', 'click']
# Step 1: Lowercase()
# Step 2: 
# initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# initialize stopwords
stop_words = set(stopwords.words('english'))

corpus = []


for i in range(0, df.shape[0]):
    # Applying RE to process the messages..
    msg = df['message'][i] # Get the current record message
    msg = msg.lower() # Step 1: lowercase the essage content
    msg = fix(msg) # Step 2: Expand contractions
    msg = re.sub(r'\b[\w\-.]+?@\w+?\.\w{2,4}\b', 'emailaddr', msg) # replace the actual email address
    msg = re.sub(r'(http[s]?\S+)|(\w+\.[A-Za-z]{2,4}\S*)', 'httpaddr', msg) # replacee the http address or any website link
    msg = re.sub(r'£|\$', 'moneysymb', msg) # Replace the money symbols.. 
    msg = re.sub(r'\b(\+\d{1,2}\s)?\d?[\-(.]?\d{3}\)?[\s.-]?\d{3}[\s.-]?\d{4}\b', 'phonenumbr', msg) # Phone number..
    msg = re.sub(r'\d+(\.\d+)?', 'numbr', msg) # Any number..

    # Step 4: Replace spammy keywords *before* removing punctuation
    for word in spam_words:
        msg = re.sub(fr'\b{word}\b', 'spamword', msg, flags=re.IGNORECASE)
    

    # Step 5: Remove punctuations:
    msg = re.sub('[^\w\d\s]', ' ', msg)
    msg = re.sub(' +', ' ', msg).strip() # clean extra spaces..

    # Step 6: Stopword removal
    msg = ' '.join([word for word in msg.split() if word not in stop_words])
    
    # Step 7: Lemmatization
    msg = ' '.join([lemmatizer.lemmatize(word) for word in msg.split()])

    msg = ' '.join(msg.split())
    
    # Append the cleaned message to corpus
    corpus.append(msg)

<>:31: SyntaxWarning: invalid escape sequence '\w'
<>:31: SyntaxWarning: invalid escape sequence '\w'
C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_33468\3825769654.py:31: SyntaxWarning: invalid escape sequence '\w'
  msg = re.sub('[^\w\d\s]', ' ', msg)


In [37]:
for record in corpus:
    print(record)
    

go jurong point crazy available bugis n great world la e buffet cine got amore wat
ok lar joking wif oni
spamword entry numbr wkly comp win fa cup final tkts numbrst may numbr text fa numbr receive entry question std txt rate c apply numbrovernumbr
dun say early hor c already say
nah think go usf life around though
freemsg hey darling numbr week word back would like fun still tb ok xxx std chgs send åmoneysymbnumbr rcv
even brother like speak treat like aid patent
per request melle melle oru minnaminunginte nurungu vettam set callertune caller press numbr copy friend callertune
spamword valued network customer selected receivea åmoneysymbnumbr prize reward spamword call phonenumbr spamword code klnumbr spamword numbr hour
mobile numbr month r entitled update latest colour mobile camera spamword call mobile update co spamword phonenumbr
going home soon want talk stuff anymore tonight k cried enough today
six chance win cash numbr numbr numbr pound txt cshnumbr send numbr cost numbrp day

# Word Embeddings:

In [39]:
cv = CountVectorizer()
X = cv.fit_transform(corpus).toarray()

# Ensuring categorcal target

In [43]:

y = df['target']
print (y.value_counts())

print(y[0])
print(y[1])


target
0    4825
1     747
Name: count, dtype: int64
0
0


In [45]:
#Encoding (ensuring)
le = LabelEncoder()
y = le.fit_transform(y)

print(y[0])
print(y[1])

0
0


# Splitting Training and testing

In [47]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size= 0.20, random_state = 0)

# Applying Guassian Naive` Bayes

In [56]:
bayes_classifier = MultinomialNB()
bayes_classifier.fit(X_train, y_train)


MultinomialNB()

# Predicting:

In [57]:
# Predicting
y_pred = bayes_classifier.predict(X_test)

# Results: Confusion Matrix

In [58]:
# Evaluating
cm = confusion_matrix(y_test, y_pred)

In [59]:
cm

array([[932,  17],
       [  9, 157]])

In [60]:
print ("Accuracy : %0.5f \n\n" % accuracy_score(y_test, bayes_classifier.predict(X_test)))
print (classification_report(y_test, bayes_classifier.predict(X_test)))


Accuracy : 0.97668 


              precision    recall  f1-score   support

           0       0.99      0.98      0.99       949
           1       0.90      0.95      0.92       166

    accuracy                           0.98      1115
   macro avg       0.95      0.96      0.95      1115
weighted avg       0.98      0.98      0.98      1115



# Save the model

In [ ]:
# Save the model
joblib.dump(bayes_classifier, 'models/spam_model.pkl')

# Save the CountVectorizer 
joblib.dump(cv, 'models/vectorizer.pkl')

['vectorizer.pkl']